In [1]:
import warnings, os
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from transformers import pipeline

In [2]:
torch.cuda.empty_cache()
print(torch.cuda.is_available())   # True olmalı
print(torch.cuda.device_count())  # En az 1 olmalı
print(torch.cuda.get_device_name(0))  # GPU adını verir

True
1
NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
model_id = "openai/whisper-small"
save_path = "../models/whisper-small"

required_files = ["model.safetensors", "config.json", "special_tokens_map.json"]
if not all(os.path.exists(os.path.join(save_path, file)) for file in required_files):
    print("Modelin bazı dosyaları eksik, indiriliyor...")
    # Modeli indir
    model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id)
    processor = AutoProcessor.from_pretrained(model_id)

    # DİKKAT: Hem model hem processor ayrı ayrı kaydedilmeli
    model.save_pretrained(save_path)
    processor.save_pretrained(save_path)

In [4]:
local_dir = "../models/whisper-small"

# Model ve processor'ı lokalden yükle
model = WhisperForConditionalGeneration.from_pretrained(local_dir)
processor = WhisperProcessor.from_pretrained(local_dir)

# tokenizer ve fe çalışmazsa aşağıdaki şekliyle kullanılabilir.
# from transformers import WhisperTokenizer, WhisperFeatureExtractor
# tokenizer = WhisperTokenizer.from_pretrained(local_dir)
# tokenizer.set_prefix_tokens(language="turkish")
# fe = WhisperFeatureExtractor.from_pretrained(local_dir)

tokenizer = processor.tokenizer
tokenizer.set_prefix_tokens(language="turkish")
fe = processor.feature_extractor

# Pipeline ile çalıştır (isteğe bağlı)
pipe = pipeline("automatic-speech-recognition",
                model=model,
                tokenizer=tokenizer,
                feature_extractor=fe,
                device=0 if torch.cuda.is_available() else -1,
                generate_kwargs={"language": "<|tr|>", "task": "transcribe"}
                )

# Ses dosyasını transkribe et
result = pipe("../../Downloads/hbdi.wav")

with open("hbdi.txt", "w") as file:
    file.write(result["text"])

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
